## 💻 Ambiente de Execução e Treinamento

O pipeline de treinamento e execução deste notebook foram executados e validados em ambiente de nuvem, garantindo a reprodutibilidade dos resultados em hardwares com restrição de memória.

**Especificações da Infraestrutura:**
* **Plataforma:** Kaggle Notebooks
* **Acelerador de Hardware:** NVIDIA Tesla T4 GPU (16GB VRAM)
* **Framework Base:** PyTorch com `open_clip_torch`
* **Processamento:** CUDA (`cuda:0`) ativado para aceleração de tensores.

**Resumo do Dataset Estático:**
* **Treino:** 138.000 amostras embaralhadas (Real / Fake)
* **Validação:** 12.000 amostras (Real / Fake)

**Resultados do Estudo de Ablação (1 Época):**
Os modelos foram treinados sob as mesmas condições para avaliar o impacto de cada componente da arquitetura híbrida:

| Variante do Modelo | Uso de Prompt | Uso de LBP | Acurácia de Validação | Tempo de Treino |
| :--- | :---: | :---: | :---: | :--- |
| **1. Completo (Proposto)** | ✅ | ✅ | **92.17%** | ~ 2h 19m |
| **2. Sem Prompt Tuning** | ❌ | ✅ | 91.62% | ~ 2h 15m |
| **3. Sem Extração LBP** | ✅ | ❌ | 91.58% | ~ 1h 54m |

## **Importações, clone do repositório e Configurações Iniciais:**

In [1]:
import os
import sys
import gc
import glob
import random
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

os.environ["HF_DATASETS_CACHE"] = "/kaggle/working/datasets_cache"

# Clone do repositório e instalação das dependências
!git clone https://github.com/Jvfrostbr/ClipBased-SyntheticImageDetection.git
!pip install hf_xet open_clip_torch

# Configuração dos caminhos do sistema para o Python enxergar a pasta 'networks'
%cd /kaggle/working/ClipBased-SyntheticImageDetection
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from networks.clip_custom_detector import CLIPImageClassifier

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Ambiente configurado com sucesso! Dispositivo: {DEVICE}")

class EstaticForensicDataset(Dataset):
    def __init__(self, samples, preprocess_fn):
        self.samples = samples
        self.preprocess_fn = preprocess_fn

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        amostra = self.samples[idx]
        try:
            image = Image.open(amostra['file_path']).convert('RGB')
            pixel_values = self.preprocess_fn(image)
        except Exception:
            # Fallback seguro para arquivos corrompidos
            pixel_values = torch.zeros((3, 224, 224))
            
        return pixel_values, amostra['label']

Cloning into 'ClipBased-SyntheticImageDetection'...
remote: Enumerating objects: 133, done.
remote: Counting objects: 100% (75/75), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 133 (delta 39), reused 48 (delta 27), pack-reused 58 (from 1)
Receiving objects: 100% (133/133), 597.93 KiB | 5.59 MiB/s, done.
Resolving deltas: 100% (50/50), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.4 MB/s eta 0:00:00
/kaggle/working/ClipBased-SyntheticImageDetection
🖥️ Ambiente configurado com sucesso! Dispositivo: cuda


## **Mapeamento do dataset:**

In [2]:
# ⚠️ ATENÇÃO: Substitua pelo nome da pasta que o Kaggle criar no upload
DATASET_ROOT = "/kaggle/input/datasets/jvfrostbr/dataset-train"

# Caminhos absolutos das subpastas
dir_treino_real = os.path.join(DATASET_ROOT, "treino/0_real/**/*.jpg")
dir_treino_fake = os.path.join(DATASET_ROOT, "treino/1_fake/**/*.jpg")
dir_val_real = os.path.join(DATASET_ROOT, "validacao/0_real/**/*.jpg")
dir_val_fake = os.path.join(DATASET_ROOT, "validacao/1_fake/**/*.jpg")

amostras_treino = []
amostras_val = []

# Mapeia o Treino
for path in glob.glob(dir_treino_real, recursive=True):
    amostras_treino.append({'file_path': path, 'label': 0})
for path in glob.glob(dir_treino_fake, recursive=True):
    amostras_treino.append({'file_path': path, 'label': 1})

# Mapeia a Validação
for path in glob.glob(dir_val_real, recursive=True):
    amostras_val.append({'file_path': path, 'label': 0})
for path in glob.glob(dir_val_fake, recursive=True):
    amostras_val.append({'file_path': path, 'label': 1})

# Embaralhamento global obrigatório
random.shuffle(amostras_treino)
random.shuffle(amostras_val)

print("="*50)
print("📊 RESUMO DO DATASET ESTÁTICO")
print("="*50)
print(f"Amostras de TREINO: {len(amostras_treino)}")
print(f"Amostras de VALIDAÇÃO: {len(amostras_val)}")

📊 RESUMO DO DATASET ESTÁTICO
Amostras de TREINO: 138000
Amostras de VALIDAÇÃO: 12000


## **Instanciação dos Datasets no PyTorch:**

In [3]:
print("📦 Inicializando objetos do Dataset...")

# Importamos o modelo temporário só para puxar o processador de imagem correto
IS_MULTICLASS = False
model_temp = CLIPImageClassifier(use_prompt=False, use_lbp=False, multiclass=IS_MULTICLASS)

train_dataset = EstaticForensicDataset(amostras_treino, model_temp.preprocess)
val_dataset = EstaticForensicDataset(amostras_val, model_temp.preprocess)

del model_temp
gc.collect()

print("✅ Tudo pronto! Pode rodar a função de treinamento.")

📦 Inicializando objetos do Dataset...


open_clip_model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


✅ Tudo pronto! Pode rodar a função de treinamento.


## **Pipeline de Treinamento e Validação**

In [4]:
# --- FUNÇÃO DE TREINAMENTO E VALIDAÇÃO ADAPTADA PARA PYTORCH PURO ---

def custom_collate(batch):
    """ Monta os batches garantindo o empacotamento em Tensores limpos (Índices 0 e 1) """
    images = torch.stack([item[0] for item in batch])
    labels = torch.tensor([item[1] for item in batch])
    return images, labels

def treinar_variacao(config, train_set, val_set, device, epochs=5, batch_size=64, lr=1e-4):
    print(f"\n" + "="*60)
    print(f"🤖 INICIANDO EXPERIMENTO: {config['nome'].upper()}")
    print("="*60 + "\n")

    # Instanciação oficial usando a nova classe e a flag de multiclasse
    model = CLIPImageClassifier(
        use_prompt=config['prompt'], 
        use_lbp=config['lbp'],
        multiclass=config['multiclass']
    ).to(device)
    
    # Configuração do Otimizador AdamW (Prompt Tuning vs MLP pura)
    if config['prompt']:
        optimizer = optim.AdamW([
            {'params': [model.soft_prompts], 'lr': lr},
            {'params': model.mlp.parameters(), 'lr': lr}
        ], weight_decay=0.01)
    else:
        optimizer = optim.AdamW(model.mlp.parameters(), lr=lr, weight_decay=0.01)

    # Configuração dinâmica da perda com base na flag multiclass
    criterion = nn.BCEWithLogitsLoss() if not config['multiclass'] else nn.CrossEntropyLoss()

    # shuffle=True de volta ao jogo com leitura direta
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, collate_fn=custom_collate, num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=2)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, collate_fn=custom_collate, num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=2)
    WEIGHTS_DIR = f"/kaggle/working/weights/{config['nome']}"
    os.makedirs(WEIGHTS_DIR, exist_ok=True)

    # Loop principal de treinamento por épocas
    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        
        pbar = tqdm(train_loader, desc=f"Época [{epoch+1}/{epochs}] - Treino")
        for images, labels in pbar:
            images = images.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            
            # Desvio dinâmico do cálculo de perda e acurácia baseados no escopo da rede
            if not config['multiclass']:
                labels_dev = labels.float().unsqueeze(1).to(device)
                loss = criterion(outputs, labels_dev)
                preds = (torch.sigmoid(outputs) > 0.5).float()
            else:
                labels_dev = labels.long().to(device)
                loss = criterion(outputs, labels_dev)
                preds = outputs.argmax(dim=-1)
                
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            correct += (preds == labels_dev).sum().item()
            total += labels_dev.size(0)
            
            pbar.set_postfix(loss=(running_loss / (total/batch_size + 1e-5)), acc=(correct / total))
            
        # Avaliação em tempo de execução no conjunto de validação
        model.eval()
        val_correct, val_total = 0, 0
        bin_correct, bin_total = 0, 0 
        
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                outputs = model(images)
                
                if not config['multiclass']:
                    labels_dev = labels.float().unsqueeze(1).to(device)
                    preds = (torch.sigmoid(outputs) > 0.5).float()
                    val_correct += (preds == labels_dev).sum().item()
                    val_total += labels_dev.size(0)
                else:
                    labels_dev = labels.long().to(device)
                    preds = outputs.argmax(dim=-1)
                    val_correct += (preds == labels_dev).sum().item()
                    val_total += labels_dev.size(0)
                    
                    # Isolando o território binário (0 vs 1) dentro do modelo multiclasse para monitoramento
                    bin_mask = (labels_dev == 0) | (labels_dev == 1)
                    if bin_mask.any():
                        bin_logits = outputs[bin_mask][:, :2] 
                        bin_labels = labels_dev[bin_mask]
                        bin_preds = bin_logits.argmax(dim=-1)
                        bin_correct += (bin_preds == bin_labels).sum().item()
                        bin_total += bin_labels.size(0)

        # Print das métricas da época correspondente
        epoch_val_acc = val_correct / (val_total + 1e-5)
        if config['multiclass']:
            epoch_bin_acc = bin_correct / (bin_total + 1e-5)
            print(f"📈 Val Acc Geral (3C): {epoch_val_acc:.4f} | Subconjunto Binário (0 vs 1): {epoch_bin_acc:.4f}")
        else:
            print(f"📈 Val Acc: {epoch_val_acc:.4f}")

        # Salvamento completo do Checkpoint estruturado (Pesos + Estado do Otimizador)
        save_path = os.path.join(WEIGHTS_DIR, f"checkpoint_ep{epoch+1}.pth")
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': epoch_val_acc
        }, save_path)

    # Coletor de lixo e limpeza de cache da GPU para evitar vazamento de VRAM entre variações
    del model, optimizer, train_loader, val_loader
    gc.collect()
    torch.cuda.empty_cache()

# **Inicialização do treinamento**

In [5]:
# --- CONFIGURAÇÃO E EXECUÇÃO DO EXPERIMENTO DA VEZ ---

experimentos = {
    "v1": {"nome": "1_completo",       "prompt": True,  "lbp": True,  "multiclass": False},
    "v2": {"nome": "2_sem_prompt",     "prompt": False, "lbp": True,  "multiclass": False},
    "v3": {"nome": "3_sem_lbp",        "prompt": True,  "lbp": False, "multiclass": False},
}

# Loop para rodar as 3 variantes de uma vez só na mesma sessão
for chave, exp_atual in experimentos.items():
    
    # Chamada do pipeline apontando para os objetos estáveis gerados na Célula 2
    treinar_variacao(exp_atual, train_dataset, val_dataset, DEVICE, epochs=1, batch_size=64)
    print(f"\n✨ [VARIANTE CONCLUÍDA] {exp_atual['nome']} finalizada com sucesso!\n")
    
print("\n🏁 [FIM GLOBAL] Todas as variantes do estudo de ablação foram treinadas sob o mesmo dataset estável!")


🤖 INICIANDO EXPERIMENTO: 1_COMPLETO



Época [1/1] - Treino: 100%|██████████| 2157/2157 [2:19:18<00:00,  3.88s/it, acc=0.915, loss=0.205]


📈 Val Acc: 0.9217

✨ [VARIANTE CONCLUÍDA] 1_completo finalizada com sucesso!


🤖 INICIANDO EXPERIMENTO: 2_SEM_PROMPT



Época [1/1] - Treino: 100%|██████████| 2157/2157 [2:15:01<00:00,  3.76s/it, acc=0.91, loss=0.217]


📈 Val Acc: 0.9162

✨ [VARIANTE CONCLUÍDA] 2_sem_prompt finalizada com sucesso!


🤖 INICIANDO EXPERIMENTO: 3_SEM_LBP



Época [1/1] - Treino: 100%|██████████| 2157/2157 [1:54:26<00:00,  3.18s/it, acc=0.914, loss=0.21]


📈 Val Acc: 0.9158

✨ [VARIANTE CONCLUÍDA] 3_sem_lbp finalizada com sucesso!


🏁 [FIM GLOBAL] Todas as variantes do estudo de ablação foram treinadas sob o mesmo dataset estável!
